In [18]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="indobenchmark/indobert-base-p1")
print(classifier("IKN bagus! Saya dukung pembangunan IKN"))

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0


[{'label': 'LABEL_1', 'score': 0.3095991611480713}]


In [14]:
classifier = pipeline("sentiment-analysis", model="indolem/indobert-base-uncased")
print(classifier("IKN sangat bagus"))

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indolem/indobert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0


[{'label': 'LABEL_0', 'score': 0.5772546529769897}]


In [15]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="w11wo/indonesian-roberta-base-sentiment-classifier")
print(classifier("IKN sangat bagus"))
print(classifier("Saya khawatir pembangunan IKN akan merusak lingkungan"))

Device set to use mps:0


[{'label': 'positive', 'score': 0.9986158609390259}]
[{'label': 'negative', 'score': 0.6055940389633179}]


In [ ]:
from transformers import pipeline

pretrained_name = "w11wo/indonesian-roberta-base-sentiment-classifier"

nlp = pipeline(
    "sentiment-analysis",
    model=pretrained_name,
    tokenizer=pretrained_name
)

nlp("Makanya dukung IKN biar ga diem2 digondol maling")

Device set to use mps:0


[{'label': 'negative', 'score': 0.9971785545349121}]

In [ ]:
from transformers import pipeline

# Load model pre-trained sentiment
pretrained_name = "w11wo/indonesian-roberta-base-sentiment-classifier"
nlp = pipeline("sentiment-analysis", model=pretrained_name, tokenizer=pretrained_name)

# Daftar keyword sederhana untuk override
support_keywords = ["dukung ikn", "setuju ikn", "mendukung ikn", "pro ikn"]
reject_keywords = ["tolak ikn", "anti ikn", "batalkan ikn", "stop ikn", "ikn mangkrak", ]
kritik_keywords = ["tapi", "namun", "sebaiknya", "perlu", "harus"]

def classify_ikn(text):
    # Step 1: model prediction
    result = nlp(text)[0]
    sentiment = result['label']
    score = result['score']
    text_low = text.lower()
    
    # Step 2: rule-based override
    if any(kw in text_low for kw in support_keywords):
        return {"label": "Dukungan", "score": result['score']}
    elif any(kw in text_low for kw in reject_keywords):
        return {"label": "Penolakan", "score": result['score']}
    elif sentiment == "neutral":
        return {"label": "Netral", "score": result['score']}
    elif sentiment == "negative" and any(kw in text_low for kw in kritik_keywords):
        return {"label": "Kritik Konstruktif", "score": result['score']}
    elif sentiment == "negative":
        return {"label": "Penolakan", "score": result['score']}
    elif sentiment == "positive":
        return {"label": "Dukungan", "score": result['score']}
    else:
        return {"label": "Netral", "score": result['score']}

# 🔎 Uji coba
samples = [
    "IKN mangkrak",
    "Makanya dukung IKN biar ga diem2 digondol maling",
    "IKN berlokasi di Kalimantan Timur",
    "Saya setuju IKN, tapi harus tetap jaga lingkungan"
]

for s in samples:
    print(s, "→", classify_ikn(s))


Device set to use mps:0


IKN mangkrak → {'label': 'Penolakan', 'score': 0.9971785545349121}
Makanya dukung IKN biar ga diem2 digondol maling → {'label': 'Dukungan', 'score': 0.9570462703704834}
IKN berlokasi di Kalimantan Timur → {'label': 'Dukungan', 'score': 0.5715645551681519}
Saya setuju IKN, tapi harus tetap jaga lingkungan → {'label': 'Dukungan', 'score': 0.7813253998756409}


In [20]:
from transformers import pipeline

# Load model pre-trained sentiment
pretrained_name = "w11wo/indonesian-roberta-base-sentiment-classifier"
nlp = pipeline("sentiment-analysis", model=pretrained_name, tokenizer=pretrained_name)

# Daftar keyword sederhana
support_keywords = ["dukung ikn", "setuju ikn", "mendukung ikn", "pro ikn"]
reject_keywords = ["tolak ikn", "anti ikn", "batalkan ikn", "stop ikn"]
kritik_keywords = ["tapi", "namun", "sebaiknya", "perlu", "harus"]

threshold = 0.7

def classify_ikn(text):
    # Prediksi model
    result = nlp(text)[0]
    sentiment = result['label']
    score = result['score']
    text_low = text.lower()

    # 🔑 Urutan prioritas aturan
    # 1. Kritik konstruktif (negative/positive tapi ada "tapi/namun/harus")
    if any(kw in text_low for kw in kritik_keywords) and ("ikn" in text_low):
        return {"label": "Kritik Konstruktif", "score": score}
    
    # 2. Penolakan eksplisit
    elif any(kw in text_low for kw in reject_keywords):
        return {"label": "Penolakan", "score": score}
    
    # 3. Dukungan eksplisit
    elif any(kw in text_low for kw in support_keywords):
        return {"label": "Dukungan", "score": score}
    
    # 4. Netral (jika model bilang neutral ATAU tidak ada indikasi pro/kontra)
    elif score < threshold:
        return {"label": "Netral", "score": score}
    
    # 5. Fallback → gunakan hasil model
    elif sentiment == "positive":
        return {"label": "Dukungan", "score": score}
    elif sentiment == "negative":
        return {"label": "Penolakan", "score": score}
    else:
        return {"label": "Netral", "score": score}

# 🔎 Uji coba
samples = [
    "IKN mangkrak",
    "Makanya dukung IKN biar ga diem2 digondol maling",
    "IKN berlokasi di Kalimantan Timur",
    "Saya setuju IKN, tapi harus tetap jaga lingkungan",
    "IKN mungkin bagus, entahlah"
]

for s in samples:
    print(s, "→", classify_ikn(s))

Device set to use mps:0


IKN mangkrak → {'label': 'Penolakan', 'score': 0.9971785545349121}
Makanya dukung IKN biar ga diem2 digondol maling → {'label': 'Dukungan', 'score': 0.9570462703704834}
IKN berlokasi di Kalimantan Timur → {'label': 'Netral', 'score': 0.5715645551681519}
Saya setuju IKN, tapi harus tetap jaga lingkungan → {'label': 'Kritik Konstruktif', 'score': 0.7813253998756409}
IKN mungkin bagus, entahlah → {'label': 'Netral', 'score': 0.6915395259857178}


In [23]:
import re

# split text
def split_text(text, max_len=250):
    sentences = re.split(r'(?<=[.!?]) +', text)
    return [s.strip() for s in sentences if s.strip()]

# priority
priority_scores = {
    "Kritik Konstruktif": 3,
    "Penolakan": 2,
    "Dukungan": 1,
    "Netral": 0
}

def classify_ikn(text):
    result = nlp(text)[0]
    sentiment = result['label']
    score = result['score']
    text_low = text.lower()

    # rules
    if any(kw in text_low for kw in reject_keywords):
        return {"label": "Penolakan", "score": score}

    elif any(kw in text_low for kw in support_keywords):
        # kalau ada juga kata kritik → kritik konstruktif
        if any(kw in text_low for kw in kritik_keywords):
            return {"label": "Kritik Konstruktif", "score": score}
        return {"label": "Dukungan", "score": score}

    elif sentiment == "neutral" or score < 0.7:
        return {"label": "Netral", "score": score}

    elif sentiment == "negative":
        # kalau ada kata kritik → kritik konstruktif
        if any(kw in text_low for kw in kritik_keywords):
            return {"label": "Kritik Konstruktif", "score": score}
        return {"label": "Penolakan", "score": score}

    elif sentiment == "positive":
        return {"label": "Dukungan", "score": score}

    return {"label": "Netral", "score": score}

def classify_long_text(text):
    chunks = split_text(text)
    results = [classify_ikn(chunk) for chunk in chunks]

    # hitung skor total
    score_count = {k: 0 for k in priority_scores.keys()}
    for r in results:
        score_count[r['label']] += priority_scores[r['label']]

    # pilih kategori dengan skor tertinggi
    final_label = max(score_count, key=score_count.get)

    return {
        "label": final_label,
        "detail": results,
        "score_summary": score_count
    }

# uji coba
samples = [
    "IKN ini bagus buat pemerataan pembangunan. Tapi sayang kalau nanti lingkungan tidak dijaga. Pemerintah perlu hati-hati.",
    "IKN harusnya dihentikan saja. Biayanya terlalu besar. Tidak ada manfaat jelas.",
    "IKN adalah proyek penting untuk masa depan bangsa. Saya sangat setuju dan bangga.",
    "IKN berlokasi di Kalimantan Timur."
]

for s in samples:
    print("\nTeks:", s)
    print(classify_long_text(s))



Teks: IKN ini bagus buat pemerataan pembangunan. Tapi sayang kalau nanti lingkungan tidak dijaga. Pemerintah perlu hati-hati.
{'label': 'Kritik Konstruktif', 'detail': [{'label': 'Dukungan', 'score': 0.9967105388641357}, {'label': 'Kritik Konstruktif', 'score': 0.9983737468719482}, {'label': 'Kritik Konstruktif', 'score': 0.8429647088050842}], 'score_summary': {'Kritik Konstruktif': 6, 'Penolakan': 0, 'Dukungan': 1, 'Netral': 0}}

Teks: IKN harusnya dihentikan saja. Biayanya terlalu besar. Tidak ada manfaat jelas.
{'label': 'Penolakan', 'detail': [{'label': 'Kritik Konstruktif', 'score': 0.9979215264320374}, {'label': 'Penolakan', 'score': 0.9709922671318054}, {'label': 'Penolakan', 'score': 0.984370231628418}], 'score_summary': {'Kritik Konstruktif': 3, 'Penolakan': 4, 'Dukungan': 0, 'Netral': 0}}

Teks: IKN adalah proyek penting untuk masa depan bangsa. Saya sangat setuju dan bangga.
{'label': 'Dukungan', 'detail': [{'label': 'Netral', 'score': 0.8637287616729736}, {'label': 'Dukung

In [24]:
import re

# split text
def split_text(text, max_len=250):
    sentences = re.split(r'(?<=[.!?]) +', text)
    return [s.strip() for s in sentences if s.strip()]

# priority
priority_scores = {
    "Kritik Konstruktif": 3,
    "Penolakan": 2,
    "Dukungan": 1,
    "Netral": 0
}

def classify_ikn(text):
    result = nlp(text)[0]
    sentiment = result['label']
    score = result['score']
    text_low = text.lower()

    # rules
    if any(kw in text_low for kw in reject_keywords):
        return {"label": "Penolakan", "score": score}

    elif any(kw in text_low for kw in support_keywords):
        # kalau ada juga kata kritik → kritik konstruktif
        if any(kw in text_low for kw in kritik_keywords):
            return {"label": "Kritik Konstruktif", "score": score}
        return {"label": "Dukungan", "score": score}

    elif sentiment == "neutral" or score < 0.7:
        return {"label": "Netral", "score": score}

    elif sentiment == "negative":
        # kalau ada kata kritik → kritik konstruktif
        if any(kw in text_low for kw in kritik_keywords):
            return {"label": "Kritik Konstruktif", "score": score}
        return {"label": "Penolakan", "score": score}

    elif sentiment == "positive":
        return {"label": "Dukungan", "score": score}

    return {"label": "Netral", "score": score}

def classify_long_text(text):
    chunks = split_text(text)
    results = [classify_ikn(chunk) for chunk in chunks]

    # hitung jumlah kategori
    count = {k: 0 for k in priority_scores.keys()}
    for r in results:
        count[r['label']] += 1

    # hitung skor total (jumlah × prioritas)
    score_count = {k: count[k] * priority_scores[k] for k in count}

    # pilih kategori dengan skor tertinggi
    final_label = max(score_count, key=score_count.get)

    return {
        "label": final_label,
        "detail": results,
        "count_summary": count,
        "score_summary": score_count
    }


# uji coba
samples = [
    "IKN ini bagus buat pemerataan pembangunan. Tapi sayang kalau nanti lingkungan tidak dijaga. Pemerintah perlu hati-hati.",
    "IKN harusnya dihentikan saja. Biayanya terlalu besar. Tidak ada manfaat jelas.",
    "IKN adalah proyek penting untuk masa depan bangsa. Saya sangat setuju dan bangga.",
    "IKN berlokasi di Kalimantan Timur."
]

for s in samples:
    print("\nTeks:", s)
    print(classify_long_text(s))



Teks: IKN ini bagus buat pemerataan pembangunan. Tapi sayang kalau nanti lingkungan tidak dijaga. Pemerintah perlu hati-hati.
{'label': 'Kritik Konstruktif', 'detail': [{'label': 'Dukungan', 'score': 0.9967105388641357}, {'label': 'Kritik Konstruktif', 'score': 0.9983737468719482}, {'label': 'Kritik Konstruktif', 'score': 0.8429647088050842}], 'count_summary': {'Kritik Konstruktif': 2, 'Penolakan': 0, 'Dukungan': 1, 'Netral': 0}, 'score_summary': {'Kritik Konstruktif': 6, 'Penolakan': 0, 'Dukungan': 1, 'Netral': 0}}

Teks: IKN harusnya dihentikan saja. Biayanya terlalu besar. Tidak ada manfaat jelas.
{'label': 'Penolakan', 'detail': [{'label': 'Kritik Konstruktif', 'score': 0.9979215264320374}, {'label': 'Penolakan', 'score': 0.9709922671318054}, {'label': 'Penolakan', 'score': 0.984370231628418}], 'count_summary': {'Kritik Konstruktif': 1, 'Penolakan': 2, 'Dukungan': 0, 'Netral': 0}, 'score_summary': {'Kritik Konstruktif': 3, 'Penolakan': 4, 'Dukungan': 0, 'Netral': 0}}

Teks: IKN ad

In [ ]:
import re

# split text
def split_text(text, max_len=250):
    sentences = re.split(r'(?<=[.!?]) +', text)
    return [s.strip() for s in sentences if s.strip()]

# priority
priority_scores = {
    "Kritik Konstruktif": 3,
    "Penolakan": 2,
    "Dukungan": 1,
    "Netral": 0
}

def classify_ikn(text):
    result = nlp(text)[0]
    sentiment = result['label']
    score = result['score']
    text_low = text.lower()

    # rules
    if any(kw in text_low for kw in reject_keywords):
        return {"label": "Penolakan", "score": score}

    elif any(kw in text_low for kw in support_keywords):
        # kalau ada juga kata kritik → kritik konstruktif
        if any(kw in text_low for kw in kritik_keywords):
            return {"label": "Kritik Konstruktif", "score": score}
        return {"label": "Dukungan", "score": score}

    elif sentiment == "neutral" or score < 0.7:
        return {"label": "Netral", "score": score}

    elif sentiment == "negative":
        # kalau ada kata kritik → kritik konstruktif
        if any(kw in text_low for kw in kritik_keywords):
            return {"label": "Kritik Konstruktif", "score": score}
        return {"label": "Penolakan", "score": score}

    elif sentiment == "positive":
        return {"label": "Dukungan", "score": score}

    return {"label": "Netral", "score": score}

def classify_long_text(text):
    chunks = split_text(text)
    results = [classify_ikn(chunk) for chunk in chunks]

    # hitung jumlah kategori
    count = {k: 0 for k in priority_scores.keys()}
    for r in results:
        count[r['label']] += 1

    # safeguard: kalau hanya ada 1 kategori unik di detail → ambil langsung
    unique_labels = [k for k, v in count.items() if v > 0]
    if len(unique_labels) == 1:
        return {
            "label": unique_labels[0],
            "detail": results,
            "count_summary": count,
            "score_summary": {k: count[k] * priority_scores[k] for k in count}
        }

    # hitung skor total (jumlah × prioritas)
    score_count = {k: count[k] * priority_scores[k] for k in count}

    # pilih kategori dengan skor tertinggi
    final_label = max(score_count, key=score_count.get)

    return {
        "label": final_label,
        "detail": results,
        "count_summary": count,
        "score_summary": score_count
    }


# uji coba
samples = [
    "IKN ini bagus buat pemerataan pembangunan. Tapi sayang kalau nanti lingkungan tidak dijaga. Pemerintah perlu hati-hati.",
    "IKN harusnya dihentikan saja. Biayanya terlalu besar. Tidak ada manfaat jelas.",
    "IKN adalah proyek penting untuk masa depan bangsa. Saya sangat setuju dan bangga.",
    "IKN berlokasi di Kalimantan Timur."
]

for s in samples:
    print("\nTeks:", s)
    print(classify_long_text(s))



Teks: IKN ini bagus buat pemerataan pembangunan. Tapi sayang kalau nanti lingkungan tidak dijaga. Pemerintah perlu hati-hati.
{'label': 'Kritik Konstruktif', 'detail': [{'label': 'Dukungan', 'score': 0.9967105388641357}, {'label': 'Kritik Konstruktif', 'score': 0.9983737468719482}, {'label': 'Kritik Konstruktif', 'score': 0.8429647088050842}], 'count_summary': {'Kritik Konstruktif': 2, 'Penolakan': 0, 'Dukungan': 1, 'Netral': 0}, 'score_summary': {'Kritik Konstruktif': 6, 'Penolakan': 0, 'Dukungan': 1, 'Netral': 0}}

Teks: IKN harusnya dihentikan saja. Biayanya terlalu besar. Tidak ada manfaat jelas.
{'label': 'Penolakan', 'detail': [{'label': 'Kritik Konstruktif', 'score': 0.9979215264320374}, {'label': 'Penolakan', 'score': 0.9709922671318054}, {'label': 'Penolakan', 'score': 0.984370231628418}], 'count_summary': {'Kritik Konstruktif': 1, 'Penolakan': 2, 'Dukungan': 0, 'Netral': 0}, 'score_summary': {'Kritik Konstruktif': 3, 'Penolakan': 4, 'Dukungan': 0, 'Netral': 0}}

Teks: IKN ad

In [6]:
from transformers import pipeline
import re

# Load model pre-trained sentiment
pretrained_name = "w11wo/indonesian-roberta-base-sentiment-classifier"
nlp = pipeline("sentiment-analysis", model=pretrained_name, tokenizer=pretrained_name)

# split text
def split_text(text, max_len=250):
    sentences = re.split(r'(?<=[.!?]) +', text)
    return [s.strip() for s in sentences if s.strip()]

# keywords
positive_keywords = ["dukung ikn", "setuju ikn", "mendukung ikn", "pro ikn"]
negative_keywords = ["tolak ikn", "anti ikn", "batalkan ikn", "stop ikn", "julid", "ga guna", "tidak berguna", "mahal"]
netral_keywords = ["tapi", "namun", "sebaiknya", "perlu", "harus"]

# priority
priority_scores = {
    "Positif": 1,
    "Negatif": 1,
    "Netral": 0
}

def classify_ikn(text):
    result = nlp(text)[0]
    sentiment = result['label']
    score = result['score']
    text_low = text.lower()

    # rules
    if any(kw in text_low for kw in negative_keywords):
        return {"label": "Negatif", "score": score}

    elif any(kw in text_low for kw in positive_keywords):
        return {"label": "Positif", "score": score}

    elif sentiment == "neutral" or score < 0.7:
        return {"label": "Netral", "score": score}

    elif sentiment == "negative":
        return {"label": "Negatif", "score": score}

    elif sentiment == "positive":
        return {"label": "Positif", "score": score}

    return {"label": "Netral", "score": score}

def classify_long_text(text):
    chunks = split_text(text)
    results = [classify_ikn(chunk) for chunk in chunks]

    # hitung jumlah kategori
    count = {k: 0 for k in priority_scores.keys()}
    for r in results:
        count[r['label']] += 1

    # safeguard: kalau hanya ada 1 kategori unik di detail → ambil langsung
    unique_labels = [k for k, v in count.items() if v > 0]
    if len(unique_labels) == 1:
        return {
            "label": unique_labels[0],
            "detail": results,
            "count_summary": count,
            "score_summary": {k: count[k] * priority_scores[k] for k in count}
        }

    # hitung skor total (jumlah × prioritas)
    score_count = {k: count[k] * priority_scores[k] for k in count}

    # cari label dengan skor tertinggi
    max_score = max(score_count.values())
    top_labels = [k for k, v in score_count.items() if v == max_score]

    # kalau hanya satu label unggul → pakai itu
    if len(top_labels) == 1:
        final_label = top_labels[0]
    else:
        # kalau ada seri → ambil label dengan rata-rata confidence tertinggi
        avg_scores = {}
        for lbl in top_labels:
            lbl_scores = [r["score"] for r in results if r["label"] == lbl]
            avg_scores[lbl] = sum(lbl_scores) / len(lbl_scores) if lbl_scores else 0
        final_label = max(avg_scores, key=avg_scores.get)

    return {
        "label": final_label,
        "detail": results,
        "count_summary": count,
        "score_summary": score_count
    }

# uji coba
samples = [
    "IKN ini bagus buat pemerataan pembangunan. Tapi sayang kalau nanti lingkungan tidak dijaga. Pemerintah perlu hati-hati.",
    "IKN harusnya dihentikan saja. Biayanya terlalu besar. Tidak ada manfaat jelas.",
    "IKN adalah proyek penting untuk masa depan bangsa. Saya sangat setuju dan bangga.",
    "IKN berlokasi di Kalimantan Timur.",
    "Mahal doang tapi ga guna. Kayak IKN"
]

for s in samples:
    print("\nTeks:", s)
    print(classify_long_text(s))

Device set to use mps:0



Teks: IKN ini bagus buat pemerataan pembangunan. Tapi sayang kalau nanti lingkungan tidak dijaga. Pemerintah perlu hati-hati.
{'label': 'Negatif', 'detail': [{'label': 'Positif', 'score': 0.9967105388641357}, {'label': 'Negatif', 'score': 0.9983737468719482}, {'label': 'Negatif', 'score': 0.8429647088050842}], 'count_summary': {'Positif': 1, 'Negatif': 2, 'Netral': 0}, 'score_summary': {'Positif': 1, 'Negatif': 2, 'Netral': 0}}

Teks: IKN harusnya dihentikan saja. Biayanya terlalu besar. Tidak ada manfaat jelas.
{'label': 'Negatif', 'detail': [{'label': 'Negatif', 'score': 0.9979215264320374}, {'label': 'Negatif', 'score': 0.9709922671318054}, {'label': 'Negatif', 'score': 0.984370231628418}], 'count_summary': {'Positif': 0, 'Negatif': 3, 'Netral': 0}, 'score_summary': {'Positif': 0, 'Negatif': 3, 'Netral': 0}}

Teks: IKN adalah proyek penting untuk masa depan bangsa. Saya sangat setuju dan bangga.
{'label': 'Positif', 'detail': [{'label': 'Netral', 'score': 0.8637287616729736}, {'lab